# 04 — Demand Forecasting and Model Registration
Compares ETS, Holt-Winters, and SARIMA on the same 14-day holdout, retrains the selected Holt-Winters model on all available daily data, and registers a deployment-friendly MLflow PyFunc model in Unity Catalog.

**Input:** `pgcb.gold.daily_demand`  
**Registered model:** `pgcb.models.holt_winters_demand`  
**Alias:** `champion`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.pyfunc

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from mlflow.models import infer_signature
from mlflow import MlflowClient

DAILY_DEMAND_TABLE = "pgcb.gold.daily_demand"
EXPERIMENT_NAME = "/Shared/pgcb-demand-forecasting"
REGISTERED_MODEL_NAME = "pgcb.models.holt_winters_demand"
MODEL_ALIAS = "champion"
FORECAST_HORIZON = 14

mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.set_registry_uri("databricks-uc")

## 1. Load and validate the daily series

In [ ]:
spark_df = (
    spark.table(DAILY_DEMAND_TABLE)
    .select("date", "avg_demand_mw")
    .orderBy("date")
)

df = spark_df.toPandas()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

full_dates = pd.date_range(df.index.min(), df.index.max(), freq="D")
missing_dates = full_dates.difference(df.index)
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"Missing dates before reindexing: {len(missing_dates)}")

series = df["avg_demand_mw"].asfreq("D")
series = series.interpolate(method="time").ffill().bfill().astype(float)
print(f"Model observations: {len(series):,}")

## 2. Time-based train/test split and metrics

In [ ]:
h = FORECAST_HORIZON
train = series.iloc[:-h]
test = series.iloc[-h:]

print(f"Train: {train.index.min().date()} to {train.index.max().date()} ({len(train):,} rows)")
print(f"Test:  {test.index.min().date()} to {test.index.max().date()} ({len(test):,} rows)")

def calculate_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "MAPE": np.mean(np.abs((actual - predicted) / actual)) * 100,
    }

## 3. Fit candidate models

In [ ]:
# ETS: additive trend + additive weekly seasonality
ets_fit = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="add",
    seasonal_periods=7,
).fit()
ets_forecast = ets_fit.forecast(h)
ets_metrics = calculate_metrics(test, ets_forecast)

# Holt-Winters: additive trend + multiplicative weekly seasonality
hw_fit = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="mul",
    seasonal_periods=7,
).fit()
hw_forecast = hw_fit.forecast(h)
hw_metrics = calculate_metrics(test, hw_forecast)

# SARIMA: fixed baseline specification used for this project comparison
sarima_fit = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)
sarima_forecast = sarima_fit.forecast(steps=h)
sarima_metrics = calculate_metrics(test, sarima_forecast)

comparison = pd.DataFrame({
    "ETS": ets_metrics,
    "Holt_Winters": hw_metrics,
    "SARIMA": sarima_metrics,
}).T.sort_values("MAPE")

display(comparison)

## 4. Plot the selected model against the holdout

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(test.index, test.values, label="Actual")
plt.plot(hw_forecast.index, hw_forecast.values, label="Holt-Winters forecast")
plt.xlabel("Date")
plt.ylabel("Demand (MW)")
plt.title("Holt-Winters Forecast vs Actual")
plt.legend()
plt.show()

## 5. Retrain Holt-Winters on the full series

In [ ]:
final_hw_fit = ExponentialSmoothing(
    series,
    trend="add",
    seasonal="mul",
    seasonal_periods=7,
).fit()

future_forecast = final_hw_fit.forecast(steps=7)
display(
    future_forecast.rename("predicted_demand_mw").reset_index(names="date")
)

## 6. Wrap the fitted model as an MLflow PyFunc
The registered model accepts a `days_ahead` integer and returns dated demand forecasts. This makes the Unity Catalog signature match the interface used by the agent.

In [ ]:
class HoltWintersForecastPyfunc(mlflow.pyfunc.PythonModel):
    def __init__(self, fitted_model):
        self.fitted_model = fitted_model

    def predict(self, context, model_input, params=None):
        if "days_ahead" not in model_input.columns:
            raise ValueError("Input must contain a 'days_ahead' column.")

        days_ahead = int(model_input.iloc[0]["days_ahead"])
        if days_ahead < 1:
            raise ValueError("days_ahead must be at least 1.")

        forecast = self.fitted_model.forecast(steps=days_ahead)
        return pd.DataFrame({
            "date": [str(idx.date()) for idx in forecast.index],
            "predicted_demand_mw": [float(v) for v in forecast.values],
        })

python_model = HoltWintersForecastPyfunc(final_hw_fit)
input_example = pd.DataFrame({"days_ahead": [1]})
output_example = python_model.predict(None, input_example)
signature = infer_signature(input_example, output_example)

print(signature)
display(output_example)

## 7. Log, register, and assign the `champion` alias

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS pgcb.models")

with mlflow.start_run(run_name="FINAL_Holt_Winters"):
    mlflow.log_params({
        "model_type": "Holt-Winters",
        "trend": "add",
        "seasonality": "mul",
        "seasonal_periods": 7,
        "validation_horizon_days": h,
    })
    mlflow.log_metrics({
        "validation_mae": float(hw_metrics["MAE"]),
        "validation_rmse": float(hw_metrics["RMSE"]),
        "validation_mape": float(hw_metrics["MAPE"]),
    })

    logged_model = mlflow.pyfunc.log_model(
        name="holt_winters_model",
        python_model=python_model,
        input_example=input_example,
        signature=signature,
    )

registered_model = mlflow.register_model(
    model_uri=logged_model.model_uri,
    name=REGISTERED_MODEL_NAME,
)

client = MlflowClient()
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias=MODEL_ALIAS,
    version=registered_model.version,
)

print(f"Registered {REGISTERED_MODEL_NAME} version {registered_model.version} as @{MODEL_ALIAS}")

## 8. Validate the registered model

In [ ]:
MODEL_URI = f"models:/{REGISTERED_MODEL_NAME}@{MODEL_ALIAS}"
registered_forecaster = mlflow.pyfunc.load_model(MODEL_URI)

test_prediction = registered_forecaster.predict(
    pd.DataFrame({"days_ahead": [3]})
)

display(test_prediction)